### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="audiology_diagnosis",
    dataset_year="1987",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5TP4R",
    download_description="""
We get the data from UCI.
wget https://archive.ics.uci.edu/static/public/8/audiology+standardized.zip && unzip audiology+standardized.zip && rm audiology+standardized.zip audiology.standardized.names && mkdir -p local-data-warehouse/audiology_diagnosis && mv audiology.standardized.data audiology.standardized.test local-data-warehouse/audiology_diagnosis/
""",
    # References
    academic_reference_bibtex="""@incollection{bareiss1990protos,
  title={Protos: An exemplar-based learning apprentice},
  author={Bareiss, E Ray and Porter, Bruce W and Wier, Craig C},
  booktitle={Machine learning},
  pages={112--127},
  year={1990},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="bareiss1990protos",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the data from UCI and merge train and test data.

- We drop the ID column as it is uninformative here.
- The target label contains groups of labels with specifications. We merge labels into groups that represent general a diagnosis. We create 3 labels: "normal", "cochlear", "other". There is likely a much better way to partition these labels, but this is most reasonable partitioning for an acutal task, going from names and my limited domain knowledge about the various diagnoses.
- We drop duplicated rows as they might introduce too much information leakage for such small data and are likely not natural but rather an artifact of the limited number of features.
- We remove one constant column (history_fullness).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="diagnosis",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="diagnosis",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np
columns = [
    "age_gt_60","air","airBoneGap","ar_c","ar_u","bone","boneAbnormal","bser","history_buzzing","history_dizziness","history_fluctuating","history_fullness","history_heredity","history_nausea","history_noise","history_recruitment","history_ringing","history_roaring","history_vomiting","late_wave_poor","m_at_2k","m_cond_lt_1k","m_gt_1k","m_m_gt_2k","m_m_sn","m_m_sn_gt_1k","m_m_sn_gt_2k","m_m_sn_gt_500","m_p_sn_gt_2k","m_s_gt_500","m_s_sn","m_s_sn_gt_1k","m_s_sn_gt_2k","m_s_sn_gt_3k","m_s_sn_gt_4k","m_sn_2_3k","m_sn_gt_1k","m_sn_gt_2k","m_sn_gt_3k","m_sn_gt_4k","m_sn_gt_500","m_sn_gt_6k","m_sn_lt_1k","m_sn_lt_2k","m_sn_lt_3k","middle_wave_poor","mod_gt_4k","mod_mixed","mod_s_mixed","mod_s_sn_gt_500","mod_sn","mod_sn_gt_1k","mod_sn_gt_2k","mod_sn_gt_3k","mod_sn_gt_4k","mod_sn_gt_500","notch_4k","notch_at_4k","o_ar_c","o_ar_u","s_sn_gt_1k","s_sn_gt_2k","s_sn_gt_4k","speech","static_normal","tymp","viith_nerve_signs","wave_V_delayed","waveform_ItoV_prolonged","indentifier", "diagnosis"
]
df = pd.read_csv(dataset_mold.path / "audiology.standardized.data", header=None, names=columns, na_values="?")
df = pd.concat([df, pd.read_csv(dataset_mold.path / "audiology.standardized.test", header=None, names=columns, na_values="?")], ignore_index=True)
print("Loaded data shape:", df.shape)

as_cat_type = list(df)
df[as_cat_type] = df[as_cat_type].astype("category")


cochlear_classes = [
    "cochlear_age_and_noise",
    "cochlear_age_plus_poss_menieres",
    "cochlear_noise_and_heredity",
    "cochlear_poss_noise",
    "cochlear_unknown",
    "possible_menieres",
    "mixed_cochlear_age_fixation",
    "mixed_cochlear_age_otitis_media",
    "mixed_cochlear_age_s_om",
    "mixed_cochlear_unk_discontinuity",
    "mixed_cochlear_unk_fixation",
    "mixed_cochlear_unk_ser_om",
    "cochlear_age",
    "mixed_poss_noise_om",
]
normal_classes = ["normal_ear"]
other_classes = [
    "acoustic_neuroma",
    "bells_palsy",
    "conductive_discontinuity",
    "conductive_fixation",
    "mixed_poss_central_om",
    "otitis_media",
    "poss_central",
    "possible_brainstem_disorder",
    "retrocochlear_unknown",
]
def map_to_diagnosis(x):
    if x in cochlear_classes:
        return "cochlear"
    elif x in normal_classes:
        return "normal"
    elif x in other_classes:
        return "other"
    else:
        raise ValueError(f"Unknown diagnosis class: {x}")
df["diagnosis"] = df["diagnosis"].apply(map_to_diagnosis).astype("category")
df = df.drop(columns=[
    "indentifier",
    "history_fullness", # constant column
])
df = df.drop_duplicates()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits


split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()